Installing Libraries

In [1]:
!pip uninstall -y transformers torchvision torch torchaudio accelerate -q

!pip install -q \
torch==2.7.1 \
torchvision==0.22.1 \
torchaudio==2.7.1 \
--index-url https://download.pytorch.org/whl/cu128

!pip install -q \
transformers==4.53.3 \
accelerate==1.8.1 \
datasets==4.0.0 \
evaluate==0.4.5 \
sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 145.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 99.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 179.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 64.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 66.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 105.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 138.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 63.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 123.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 110.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 105.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Imports
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

In [4]:
#GPU SetUp
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
Tesla T4


In [5]:
#Seed
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
#Cuda Optimization
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [7]:
MODEL_PATH = "/content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6"

SAVE_DIR = "/content/drive/MyDrive/Cross Validation/MURIL_CV"

os.makedirs(SAVE_DIR, exist_ok=True)

In [8]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
MAX_LEN = 64

In [9]:
# Tokenizer Function
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=MAX_LEN,
        padding=False
    )


In [10]:
# Verify
print(tokenizer("Climate change is real."))

{'input_ids': [104, 35518, 4828, 1121, 5731, 121, 105], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


In [11]:
# Load Dataset
english_df = pd.read_csv("/content/English_train_data.csv")
hindi_df = pd.read_csv("/content/english_to_hindi.csv")
bengali_df = pd.read_csv("/content/english_to_bengali.csv")
none_df = pd.read_csv("/content/non_cliamte_none_data.csv")
none_hindi_df = pd.read_csv("/content/english_to_hindi_nonclimate.csv")
none_bengali_df = pd.read_csv("/content/english_to_bengali_nonclimate.csv")

In [12]:
none_df = none_df.head(1000)
none_bengali_df = none_bengali_df.head(1000)
none_hindi_df = none_hindi_df.head(1000)

In [13]:
hindi = hindi_df[["hindi","label"]].copy()
hindi.columns = ["sentence","label"]

bengali = bengali_df[["bengali","label"]].copy()
bengali.columns = ["sentence","label"]

none_hindi = none_hindi_df[["hindi","label"]].copy()
none_hindi.columns = ["sentence","label"]

none_bengali = none_bengali_df[["bengali","label"]].copy()
none_bengali.columns = ["sentence","label"]

english = english_df[["sentence","label"]].copy()

none_df = none_df[["sentence","label"]].copy()

In [14]:
train_df = pd.concat(
    [
        english,
        hindi,
        bengali,
        none_df,
        none_hindi,
        none_bengali
    ],

    ignore_index=True

)

In [15]:
train_df.shape

(41313, 2)

In [16]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41313 entries, 0 to 41312
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sentence  41313 non-null  object
 1   label     41313 non-null  object
dtypes: object(2)
memory usage: 645.6+ KB


In [17]:
# Remove Missing Values
train_df = train_df.dropna()
train_df = train_df.reset_index(drop=True)

In [18]:
# Remove Duplicate Sentences
train_df = train_df.drop_duplicates(subset="sentence")
train_df = train_df.reset_index(drop=True)

In [19]:
# Shuffle Dataset
train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [20]:
# Check Dataset
print(train_df.head())

                                            sentence    label
0  You were the ice berg, and now I'm global warm...     NONE
1  नया सीबीए मामला एक चेतावनी: जलवायु परिवर्तन पर...    FAVOR
2  : Yes they are that STUPID Study blames sandwi...  AGAINST
3  निश्चिंत रहें, शराब प्रेमी, आपका पसंदीदा टिपल ...     NONE
4  Imagine if there was a climate change and Sing...     NONE


In [21]:
# Label Encoding
label2id = {
    "AGAINST":0,
    "FAVOR":1,
    "NONE":2
}
id2label = {
    0:"AGAINST",
    1:"FAVOR",
    2:"NONE"
}
train_df["label"] = train_df["label"].map(label2id)

In [22]:
# Verify labels
print(train_df["label"].value_counts())
print()
print(train_df.head())

label
2    15614
0    12610
1    12605
Name: count, dtype: int64

                                            sentence  label
0  You were the ice berg, and now I'm global warm...      2
1  नया सीबीए मामला एक चेतावनी: जलवायु परिवर्तन पर...      1
2  : Yes they are that STUPID Study blames sandwi...      0
3  निश्चिंत रहें, शराब प्रेमी, आपका पसंदीदा टिपल ...      2
4  Imagine if there was a climate change and Sing...      2


In [23]:
# Check Class Balance
print(train_df["label"].value_counts(normalize=True))

label
2    0.382424
0    0.308849
1    0.308727
Name: proportion, dtype: float64


In [24]:
# Sentence Length Analysis
lengths = train_df["sentence"].apply(
    lambda x: len(tokenizer.tokenize(str(x)))
)
print(lengths.describe())

Token indices sequence length is longer than the specified maximum sequence length for this model (1277 > 512). Running this sequence through the model will result in indexing errors


count    40829.000000
mean        24.281222
std         17.649251
min          1.000000
25%         17.000000
50%         23.000000
75%         29.000000
max       1277.000000
Name: sentence, dtype: float64


In [25]:
# Check Maximum Length
print("Maximum Tokens :", lengths.max())
print("95 Percentile :", np.percentile(lengths,95))

Maximum Tokens : 1277
95 Percentile : 39.0


In [26]:
# Final Dataset Information
print("="*60)
print("Total Samples :", len(train_df))
print()
print(train_df["label"].value_counts())
print("="*60)

Total Samples : 40829

label
2    15614
0    12610
1    12605
Name: count, dtype: int64


In [27]:
# Clean and Save Dataset
train_df.to_csv(
    "/content/drive/MyDrive/clean_train.csv",
    index=False
)

PART 3 — 5-Fold Stratified Cross Validation

In [28]:
# Stratified 5-Fold Cross Validation
NUM_FOLDS = 5

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

In [29]:
# Create Metric Lists
fold_accuracy = []
fold_precision = []
fold_recall = []
fold_f1 = []

In [30]:
# Reset MODEL_PATH to the base pre-trained model for the CV loop
MODEL_PATH = "/content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6"
print(f"MODEL_PATH reset to: {MODEL_PATH}")

MODEL_PATH reset to: /content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6


In [31]:
# Start Cross Validation Loop
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

Fold 1/5
Training Samples : 32663
Validation Samples : 8166
Fold 2/5
Training Samples : 32663
Validation Samples : 8166
Fold 3/5
Training Samples : 32663
Validation Samples : 8166
Fold 4/5
Training Samples : 32663
Validation Samples : 8166
Fold 5/5
Training Samples : 32664
Validation Samples : 8165


In [32]:
# Compute Class Weights inside every fold
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_fold["label"]),
    y=train_fold["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print(class_weights)

tensor([1.0793, 1.0797, 0.8716], device='cuda:0')


In [33]:
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(
    train_fold,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_fold,
    preserve_index=False
)

In [34]:
# Tokenize
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/32664 [00:00<?, ? examples/s]

Map:   0%|          | 0/8165 [00:00<?, ? examples/s]

In [35]:
# Remove Sentence Column
train_dataset = train_dataset.remove_columns(
    ["sentence"]
)

val_dataset = val_dataset.remove_columns(
    ["sentence"]
)

In [36]:
# Rename Label Column
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

In [37]:
# Torch Format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [38]:
# Check Dataset
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 32664
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8165
})


PART 4 — Fine-Tuning with Cross Validation

In [39]:
# Data Collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

In [40]:
# Metrics Function
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [41]:
# Weighted Trainer
from transformers import Trainer

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights
        )

        loss = loss_fn(
            logits,
            labels
        )

        return (loss, outputs) if return_outputs else loss

In [42]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./Results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    save_total_limit=1,
    report_to="none",
    seed=42
)

In [43]:
# Early Stopping
early_stop = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001
)

In [44]:
# Load a Fresh Model for Each Fold
config = AutoConfig.from_pretrained(MODEL_PATH)
config.num_labels = 3
config.label2id = label2id
config.id2label = id2label
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    config=config,
    ignore_mismatched_sizes=True
)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [46]:
import torch

# Freeze Embeddings
for param in model.bert.embeddings.parameters():
    param.requires_grad = False

In [48]:
# Start Full Cross Validation Loop (including training, evaluation, and saving)
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

    # Compute Class Weights inside every fold
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_fold["label"]),
        y=train_fold["label"]
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float
    ).to(device)

    print("Class Weights for Fold ", fold+1, ":", class_weights)

    # Convert to HuggingFace Dataset
    train_dataset = Dataset.from_pandas(
        train_fold,
        preserve_index=False
    )

    val_dataset = Dataset.from_pandas(
        val_fold,
        preserve_index=False
    )

    # Tokenize
    train_dataset = train_dataset.map(
        tokenize,
        batched=True
    )

    val_dataset = val_dataset.map(
        tokenize,
        batched=True
    )

    # Remove Sentence Column
    train_dataset = train_dataset.remove_columns(
        ["sentence"]
    )

    val_dataset = val_dataset.remove_columns(
        ["sentence"]
    )

    # Rename Label Column
    train_dataset = train_dataset.rename_column(
        "label",
        "labels"
    )

    val_dataset = val_dataset.rename_column(
        "label",
        "labels"
    )

    # Torch Format
    train_dataset.set_format("torch")
    val_dataset.set_format("torch")

    # Check Dataset (optional, for verification within the loop)
    print("Train Dataset for Fold ", fold+1, ":", train_dataset)
    print("Validation Dataset for Fold ", fold+1, ":", val_dataset)

    # Load a Fresh Model for Each Fold
    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.num_labels = 3
    config.label2id = label2id
    config.id2label = id2label

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        config=config,
        ignore_mismatched_sizes=True
    )
    model.to(device)

    # Freeze Embeddings
    for param in model.bert.embeddings.parameters():
        param.requires_grad = False

    # Create Trainer
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early_stop]
    )

    # Train
    print(f"\nStarting training for Fold {fold+1}...")
    trainer.train()
    print(f"Finished training for Fold {fold+1}.")

    # Evaluate
    print(f"\nStarting evaluation for Fold {fold+1}...")
    metrics = trainer.evaluate()
    print(f"Evaluation Metrics for Fold {fold+1}: {metrics}")

    fold_accuracy.append(metrics["eval_accuracy"])
    fold_precision.append(metrics["eval_precision"])
    fold_recall.append(metrics["eval_recall"])
    fold_f1.append(metrics["eval_f1"])

    # Save Each Fold
    fold_dir = os.path.join(
        SAVE_DIR,
        f"Fold_{fold+1}"
    )

    trainer.save_model(fold_dir)
    tokenizer.save_pretrained(fold_dir)

    print(f"Model Saved -> {fold_dir}\n")

print("="*70)
print("Cross-validation loop completed.")
print("="*70)

Fold 1/5
Training Samples : 32663
Validation Samples : 8166
Class Weights for Fold  1 : tensor([1.0793, 1.0797, 0.8716], device='cuda:0')


Map:   0%|          | 0/32663 [00:00<?, ? examples/s]

Map:   0%|          | 0/8166 [00:00<?, ? examples/s]

Train Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 32663
})
Validation Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8166
})


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/Cross Validation/Best_MURIL_MLM6 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting training for Fold 5...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.911600,0.697632,0.714636,0.718940,0.719167,0.715127
2,0.611300,0.579036,0.764727,0.766513,0.770843,0.765325
3,0.446300,0.582003,0.774158,0.784575,0.785346,0.774218
4,0.330600,0.551612,0.798040,0.799602,0.804481,0.798358
5,0.263600,0.555314,0.809308,0.810733,0.815301,0.810025


Finished training for Fold 5.

Starting evaluation for Fold 5...


Evaluation Metrics for Fold 5: {'eval_loss': 0.5553135871887207, 'eval_accuracy': 0.8093080220453154, 'eval_precision': 0.8107328135755988, 'eval_recall': 0.8153011979006398, 'eval_f1': 0.810024852635638, 'eval_runtime': 7.6991, 'eval_samples_per_second': 1060.516, 'eval_steps_per_second': 33.251, 'epoch': 5.0}
Model Saved -> /content/drive/MyDrive/Cross Validation/MURIL_CV/Fold_5

Cross-validation loop completed.


In [49]:
# Cross Validation Summary
results_df = pd.DataFrame({
    "Fold": range(1, NUM_FOLDS + 1),
    "Accuracy": fold_accuracy,
    "Precision": fold_precision,
    "Recall": fold_recall,
    "Macro_F1": fold_f1
})

print(results_df)

   Fold  Accuracy  Precision    Recall  Macro_F1
0     1  0.803208   0.804430  0.809634  0.803755
1     2  0.806760   0.806709  0.810439  0.807526
2     3  0.798310   0.800327  0.805366  0.798895
3     4  0.798800   0.798974  0.804621  0.799521
4     5  0.809308   0.810733  0.815301  0.810025


In [50]:
# Average Performance
print("="*70)
print("5-Fold Cross Validation Results")
print("="*70)

print(f"Average Accuracy : {np.mean(fold_accuracy):.4f}")
print(f"Average Precision: {np.mean(fold_precision):.4f}")
print(f"Average Recall   : {np.mean(fold_recall):.4f}")
print(f"Average Macro F1 : {np.mean(fold_f1):.4f}")

print()

print(f"Std Accuracy : {np.std(fold_accuracy):.4f}")
print(f"Std Macro F1 : {np.std(fold_f1):.4f}")

5-Fold Cross Validation Results
Average Accuracy : 0.8033
Average Precision: 0.8042
Average Recall   : 0.8091
Average Macro F1 : 0.8039

Std Accuracy : 0.0043
Std Macro F1 : 0.0044


In [51]:
# Find the Best Fold
best_fold = np.argmax(fold_f1)

print(f"\nBest Fold : {best_fold+1}")
print(f"Best Macro F1 : {fold_f1[best_fold]:.4f}")


Best Fold : 5
Best Macro F1 : 0.8100


In [52]:
# Copy the Best Model
import shutil

best_model = os.path.join(
    SAVE_DIR,
    f"Fold_{best_fold+1}"
)

final_model = os.path.join(
    SAVE_DIR,
    "Best_Model"
)

if os.path.exists(final_model):
    shutil.rmtree(final_model)

shutil.copytree(best_model, final_model)

print("Best model copied successfully!")

Best model copied successfully!


In [53]:
# Load Best Model
BEST_MODEL = os.path.join(SAVE_DIR, "Best_Model")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL)
model.to(device)
model.eval()
print("Best model loaded successfully.")

Best model loaded successfully.


In [54]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# Load Best CV Model
# ==========================================

MODEL_PATH = "/content/drive/MyDrive/Cross Validation/MURIL_CV/Best_Model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

print("Model Loaded Successfully!")

# ==========================================
# Label Mapping
# ==========================================

label_map = {
    0: "AGAINST",
    1: "FAVOR",
    2: "NONE"
}

# ==========================================
# Prediction Function
# ==========================================

def predict(text):

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

        pred = torch.argmax(outputs.logits, dim=1).item()

    return label_map[pred]

Model Loaded Successfully!


In [56]:
hindi_test = pd.read_excel("/content/Hindi_500_test data.xlsx")

print(hindi_test.head())

hindi_test["Predicted_Label"] = hindi_test["COMMENT"].apply(predict)

hindi_test.to_excel(
    "/content/Hindi_500_Predictions_CVNEWM.xlsx",
    index=False
)

print("Hindi Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  बिल नाई एक वैज्ञानिक नहीं हैं। वह एक यांत्रिक ...
1   2           ऑनलाइन कक्षाओं के लिए और कौन उपस्थित है?
2   3  शाकाहारियों को इतना परेशान करने वाला क्यों होत...
3   4                      विवरण में बिल न्ये मर्च लिंक।
4   5  बिल के प्रति इतना घृणा क्यों है? मुझे समझ नहीं...
Hindi Prediction Saved Successfully!


In [57]:
bangla_test = pd.read_excel("/content/Bangla_500_test data.xlsx")

print(bangla_test.head())

bangla_test["Predicted_Label"] = bangla_test["COMMENT"].apply(predict)

bangla_test.to_excel(
    "/content/Bangla_500_Predictions_CVNEWM.xlsx",
    index=False
)

print("Bangla Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  পৃথিবীর নৃত্য এবং অনন্ত আকাশের মধ্যে, ফিসফিস ক...
1   2  ধন্যবাদ, আপনার করা সমস্ত কিছুর জন্য এবং আপনি য...
2   3  এটি বায়ুমণ্ডলে কার্বন ডাই-অক্সাইডের ২৫০ ডিগ্র...
3   4  জলবায়ু-পরিবর্তক দেবতা-পাথর জানে যে পৃথিবীর দু...
4   5  আমার শিক্ষক আমাকে বলেছিলেন যে প্রতি ১০০ বছরে প...
Bangla Prediction Saved Successfully!
